# Fase C — Consultas SQL analíticas

**Pregunta guía del proyecto:** *¿Cuáles son los patrones temporales, geográficos y operacionales de los homicidios en Colombia que permiten identificar zonas de alto riesgo y factores para la prevención del delito?*

Seis consultas contra la bodega en PostgreSQL (cargada por `fase_b_etl.ipynb`). Cada una trae su pregunta de negocio, el SQL, el resultado en tabla y una interpretación.

| # | Pregunta | Técnica SQL |
|---|---|---|
| C1 | Evolución anual y variación interanual | `JOIN` + `LAG()` |
| C2 | Departamentos con más homicidios y cambio entre quinquenios | `JOIN` + `SUM() OVER ()` + `FILTER` |
| C3 | Municipio que concentra la violencia en cada departamento | `ROW_NUMBER() OVER (PARTITION BY …)` |
| C4 | Días y franjas horarias de mayor riesgo | `GROUP BY` día × franja |
| C5 | Perfil de víctima × zona × mecanismo | `GROUP BY` con **4 atributos de 3 dimensiones** |
| C6 | Escenarios del hecho según sexo de la víctima | `RANK() OVER (PARTITION BY …)` |

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError, ArgumentError, ProgrammingError

RAIZ = Path.cwd().parent if Path.cwd().name == "entrega" else Path.cwd()
load_dotenv(RAIZ / ".env")
DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("No se encontró DATABASE_URL. Copie .env.example a .env y ajústelo.")

try:
    engine = create_engine(DATABASE_URL, pool_pre_ping=True)
    with engine.connect() as conn:
        n = conn.execute(text("SELECT COUNT(*) FROM fact_homicidios")).scalar()
except (ArgumentError, OperationalError) as e:
    raise RuntimeError(f"No fue posible conectar a PostgreSQL con DATABASE_URL: {e}") from None
except ProgrammingError:
    raise RuntimeError("La bodega está vacía: ejecute primero entrega/fase_b_etl.ipynb") from None
print(f"Conectado ✔  fact_homicidios tiene {n:,} filas")


def consultar(sql: str) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine)

Conectado ✔  fact_homicidios tiene 125,284 filas


## C1. ¿Cómo han evolucionado los homicidios año a año entre 2015 y 2024, y cuál fue la variación frente al año anterior?

In [2]:
C1 = '''
SELECT t.anio,
       SUM(f.cantidad_homicidios)                                   AS homicidios,
       LAG(SUM(f.cantidad_homicidios)) OVER (ORDER BY t.anio)       AS homicidios_anio_anterior,
       ROUND(100.0 * (SUM(f.cantidad_homicidios)
                      - LAG(SUM(f.cantidad_homicidios)) OVER (ORDER BY t.anio))
             / LAG(SUM(f.cantidad_homicidios)) OVER (ORDER BY t.anio), 1) AS variacion_pct
FROM fact_homicidios f
JOIN dim_tiempo t ON f.sk_fecha = t.sk_fecha
GROUP BY t.anio
ORDER BY t.anio;
'''
c1 = consultar(C1)
c1

,anio,homicidios,homicidios_anio_anterior,variacion_pct
0,2015,11585,NaN,NaN
1,2016,11532,11585.0,-0.5
2,2017,11373,11532.0,-1.4
3,2018,12130,11373.0,6.7
4,2019,11880,12130.0,-2.1
5,2020,11326,11880.0,-4.7
6,2021,13238,11326.0,16.9
7,2022,13939,13238.0,5.3
8,2023,14260,13939.0,2.3
9,2024,14021,14260.0,-1.7


**Interpretación.** Entre 2015 y 2020 los homicidios se mantuvieron estables (≈11.300–12.100 por año), con 2020 como mínimo del periodo (−4,7 %), coherente con las restricciones de movilidad por la pandemia. En 2021 hubo un salto de **+16,9 %**, el mayor del decenio, y desde entonces el nivel quedó en un piso más alto (>13.900 casos en 2022–2024). En 2024 se registró la primera reducción (−1,7 %) después de tres años de alzas, pero aún un 21 % por encima de 2015.

## C2. ¿Qué departamentos concentran más homicidios, qué participación tienen en el total nacional y cómo cambiaron entre 2015–2019 y 2020–2024?

In [3]:
C2 = '''
SELECT g.departamento,
       SUM(f.cantidad_homicidios) AS total,
       ROUND(100.0 * SUM(f.cantidad_homicidios)
             / SUM(SUM(f.cantidad_homicidios)) OVER (), 1)                    AS participacion_pct,
       SUM(f.cantidad_homicidios) FILTER (WHERE t.anio BETWEEN 2015 AND 2019) AS periodo_2015_2019,
       SUM(f.cantidad_homicidios) FILTER (WHERE t.anio BETWEEN 2020 AND 2024) AS periodo_2020_2024,
       ROUND(100.0 * (SUM(f.cantidad_homicidios) FILTER (WHERE t.anio BETWEEN 2020 AND 2024)
                    - SUM(f.cantidad_homicidios) FILTER (WHERE t.anio BETWEEN 2015 AND 2019))
             / NULLIF(SUM(f.cantidad_homicidios) FILTER (WHERE t.anio BETWEEN 2015 AND 2019), 0), 1)
                                                                              AS cambio_pct
FROM fact_homicidios f
JOIN dim_geografia g ON f.sk_geografia = g.sk_geografia
JOIN dim_tiempo    t ON f.sk_fecha     = t.sk_fecha
WHERE g.departamento <> 'Sin información'
GROUP BY g.departamento
ORDER BY total DESC;
'''
c2 = consultar(C2)
print(f"Top 3 = {c2.participacion_pct.head(3).sum():.1f} % | Top 10 = {c2.participacion_pct.head(10).sum():.1f} % del total")
c2.head(15)

Top 3 = 43.2 % | Top 10 = 70.7 % del total


,departamento,total,participacion_pct,periodo_2015_2019,periodo_2020_2024,cambio_pct
0,Valle del Cauca,23491,18.8,11899,11592,-2.6
1,Antioquia,18852,15.1,9532,9320,-2.2
2,"Bogotá, D.C.",11667,9.3,6008,5659,-5.8
3,Atlántico,6607,5.3,2837,3770,32.9
4,Cauca,6258,5.0,2123,4135,94.8
5,Norte de Santander,5461,4.4,2551,2910,14.1
6,Bolívar,4930,3.9,1960,2970,51.5
7,Cundinamarca,4185,3.3,2123,2062,-2.9
8,Nariño,3660,2.9,1778,1882,5.8
9,Magdalena,3382,2.7,1094,2288,109.1


**Interpretación.** La violencia homicida está muy concentrada: **Valle del Cauca (18,8 %), Antioquia (15,1 %) y Bogotá (9,3 %)** suman más de 4 de cada 10 casos, y los diez primeros departamentos el 70,7 %. Sin embargo, los tres líderes se mantuvieron estables o bajaron levemente entre quinquenios, mientras que el aumento nacional lo explican departamentos como **Cauca (+95 %), Magdalena (+109 %), Bolívar (+52 %) y Atlántico (+33 %)**, es decir, la violencia se está desplazando hacia el Pacífico y el Caribe.

## C3. En cada departamento, ¿cuál es el municipio con más homicidios y qué porcentaje del departamento concentra?

In [4]:
C3 = '''
WITH por_municipio AS (
    SELECT g.departamento, g.municipio, SUM(f.cantidad_homicidios) AS homicidios
    FROM fact_homicidios f
    JOIN dim_geografia g ON f.sk_geografia = g.sk_geografia
    WHERE g.departamento NOT IN ('Sin información', 'Bogotá, D.C.')   -- Bogotá es un único municipio
    GROUP BY g.departamento, g.municipio
),
ranking AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY departamento ORDER BY homicidios DESC) AS puesto,
           SUM(homicidios) OVER (PARTITION BY departamento)                      AS total_departamento
    FROM por_municipio
)
SELECT departamento, municipio AS municipio_lider, homicidios, total_departamento,
       ROUND(100.0 * homicidios / total_departamento, 1) AS concentracion_pct
FROM ranking
WHERE puesto = 1
ORDER BY total_departamento DESC;
'''
c3 = consultar(C3)
c3.head(15)

,departamento,municipio_lider,homicidios,total_departamento,concentracion_pct
0,Valle del Cauca,Cali,11408,23491.0,48.6
1,Antioquia,Medellín,4729,18852.0,25.1
2,Atlántico,Barranquilla,3846,6607.0,58.2
3,Cauca,Santander de Quilichao,1049,6258.0,16.8
4,Norte de Santander,Cúcuta,2599,5461.0,47.6
5,Bolívar,Cartagena de Indias,2949,4930.0,59.8
6,Cundinamarca,Soacha,1624,4185.0,38.8
7,Nariño,San Andrés de Tumaco,1726,3660.0,47.2
8,Magdalena,Santa Marta,1422,3382.0,42.0
9,Santander,Bucaramanga,1129,3357.0,33.6


**Interpretación.** En los departamentos más violentos, un solo municipio (normalmente la capital) explica entre un tercio y más de la mitad de los casos: **Cartagena 59,8 %, Barranquilla 58,2 %, Cali 48,6 %, Cúcuta 47,6 %, Tumaco 47,2 %**. Cauca es la excepción: su municipio líder es **Santander de Quilichao con solo 16,8 %**, lo que indica una violencia dispersa y rural que exige estrategias territoriales distintas a las de las capitales. Tumaco y Soacha muestran que no siempre es la capital la que lidera.

## C4. ¿En qué días de la semana y franjas horarias se concentran los homicidios?
> Solo se consideran los registros con franja horaria informada (≈41 % del total; el resto es “Sin información”).

In [5]:
C4 = '''
SELECT t.dia_num, t.dia_semana, t.hora_orden, t.rango_hora_3h,
       SUM(f.cantidad_homicidios) AS homicidios,
       ROUND(100.0 * SUM(f.cantidad_homicidios) / SUM(SUM(f.cantidad_homicidios)) OVER (), 2) AS pct
FROM fact_homicidios f
JOIN dim_tiempo t ON f.sk_fecha = t.sk_fecha
WHERE t.dia_num > 0 AND t.hora_orden > 0
GROUP BY t.dia_num, t.dia_semana, t.hora_orden, t.rango_hora_3h
ORDER BY homicidios DESC;
'''
c4 = consultar(C4)
display(c4.head(10))
tabla_c4 = (c4.pivot_table(index=["hora_orden", "rango_hora_3h"], columns=["dia_num", "dia_semana"],
                           values="homicidios").droplevel(0).droplevel(0, axis=1))
tabla_c4

,dia_num,dia_semana,hora_orden,rango_hora_3h,homicidios,pct
0,6,Sábado,8,21:00 a 23:59,2148,4.19
1,7,Domingo,1,00:00 a 02:59,1908,3.72
2,7,Domingo,8,21:00 a 23:59,1892,3.69
3,6,Sábado,7,18:00 a 20:59,1838,3.58
4,7,Domingo,7,18:00 a 20:59,1803,3.51
5,5,Viernes,7,18:00 a 20:59,1588,3.10
6,7,Domingo,2,03:00 a 05:59,1571,3.06
7,4,Jueves,7,18:00 a 20:59,1542,3.01
8,2,Martes,7,18:00 a 20:59,1507,2.94
9,5,Viernes,8,21:00 a 23:59,1486,2.90


dia_semana,Lunes,Martes,Miércoles,Jueves,Viernes,Sábado,Domingo
rango_hora_3h,,,,,,,
00:00 a 02:59,1143.0,491.0,490.0,484.0,557.0,771.0,1908.0
03:00 a 05:59,644.0,386.0,332.0,361.0,411.0,495.0,1571.0
06:00 a 08:59,470.0,481.0,489.0,501.0,456.0,453.0,746.0
09:00 a 11:59,598.0,587.0,617.0,647.0,629.0,601.0,729.0
12:00 a 14:59,691.0,704.0,714.0,762.0,669.0,811.0,866.0
15:00 a 17:59,854.0,827.0,803.0,892.0,881.0,1000.0,1000.0
18:00 a 20:59,1474.0,1507.0,1459.0,1542.0,1588.0,1838.0,1803.0
21:00 a 23:59,1250.0,1234.0,1248.0,1311.0,1486.0,2148.0,1892.0


**Interpretación.** El riesgo se concentra en la **noche del fin de semana**: las celdas más altas son sábado 21:00–23:59, domingo 00:00–02:59 y domingo 21:00–23:59. Las franjas de 18:00 a 23:59 acumulan más homicidios que todas las franjas entre las 03:00 y las 14:59, y el domingo es el día más violento. También destaca el lunes de 00:00 a 02:59 (1.143 casos, el doble que otros días laborales), consistente con los fines de semana festivos (“puentes”). Este patrón apunta a actividades de ocio y consumo de alcohol, lo que sustenta concentrar patrullajes y controles en ese horario.

## C5. ¿Cómo se combinan el sexo y la edad de la víctima, la zona del hecho y el mecanismo causal?
Agrupa por **cuatro atributos de tres dimensiones distintas** (`dim_victima`, `dim_geografia`, `dim_modalidad`).

In [6]:
C5 = '''
SELECT v.sexo,
       v.grupo_edad_quinquenal,
       g.zona_hecho,
       m.mecanismo_causal,
       SUM(f.cantidad_homicidios) AS homicidios
FROM fact_homicidios f
JOIN dim_victima   v ON f.sk_victima   = v.sk_victima
JOIN dim_geografia g ON f.sk_geografia = g.sk_geografia
JOIN dim_modalidad m ON f.sk_modalidad = m.sk_modalidad
GROUP BY v.sexo, v.grupo_edad_quinquenal, g.zona_hecho, m.mecanismo_causal
ORDER BY homicidios DESC;
'''
c5 = consultar(C5)
print(f"{len(c5):,} combinaciones distintas")
c5.head(10)

1,111 combinaciones distintas


,sexo,grupo_edad_quinquenal,zona_hecho,mecanismo_causal,homicidios
0,Hombre,(20 a 24),Cabecera municipal,Proyectil de arma de fuego,12944
1,Hombre,(25 a 29),Cabecera municipal,Proyectil de arma de fuego,12124
2,Hombre,(30 a 34),Cabecera municipal,Proyectil de arma de fuego,9439
3,Hombre,(35 a 39),Cabecera municipal,Proyectil de arma de fuego,6568
4,Hombre,(40 a 44),Cabecera municipal,Proyectil de arma de fuego,4474
5,Hombre,(18 a 19),Cabecera municipal,Proyectil de arma de fuego,3924
6,Hombre,(25 a 29),Parte rural (vereda y campo),Proyectil de arma de fuego,3786
7,Hombre,(20 a 24),Parte rural (vereda y campo),Proyectil de arma de fuego,3765
8,Hombre,(30 a 34),Parte rural (vereda y campo),Proyectil de arma de fuego,3098
9,Hombre,(20 a 24),Cabecera municipal,Cortopunzante,3034


In [7]:
# Resumen del resultado: participación de cada mecanismo dentro de cada sexo (%)
resumen_c5 = (c5[c5.sexo != "Indeterminado"]
              .pivot_table(index="mecanismo_causal", columns="sexo", values="homicidios", aggfunc="sum")
              .fillna(0))
resumen_c5 = (100 * resumen_c5 / resumen_c5.sum()).round(1).sort_values("Hombre", ascending=False)
resumen_c5.head(6)

sexo,Hombre,Mujer
mecanismo_causal,,
Proyectil de arma de fuego,75.5,56.6
Cortopunzante,16.6,21.3
Contundente,2.7,5.8
Cortocontundente,2.0,3.3
Generadores de asfixia,1.4,9.0
Agentes y mecanismo explosivo,0.7,1.1


**Interpretación.** La combinación dominante es **hombre de 20 a 29 años, en cabecera municipal, asesinado con arma de fuego**. El arma de fuego causa el 75,5 % de las muertes de hombres, pero solo el 56,6 % de las de mujeres; en ellas pesan más el arma cortopunzante (21,3 %) y la asfixia (9,0 % frente a 1,4 % en hombres), mecanismos de contacto cercano típicos de la violencia en el entorno íntimo. En zona rural el arma de fuego es incluso más frecuente (77,6 %) que en las cabeceras (72,8 %).

## C6. ¿Cuáles son los 5 escenarios donde más ocurren homicidios de hombres y de mujeres?

In [8]:
C6 = '''
WITH escenarios AS (
    SELECT v.sexo,
           m.escenario_hecho,
           SUM(f.cantidad_homicidios) AS homicidios,
           ROUND(100.0 * SUM(f.cantidad_homicidios)
                 / SUM(SUM(f.cantidad_homicidios)) OVER (PARTITION BY v.sexo), 1) AS pct_del_sexo,
           RANK() OVER (PARTITION BY v.sexo ORDER BY SUM(f.cantidad_homicidios) DESC) AS puesto
    FROM fact_homicidios f
    JOIN dim_victima   v ON f.sk_victima   = v.sk_victima
    JOIN dim_modalidad m ON f.sk_modalidad = m.sk_modalidad
    WHERE v.sexo IN ('Hombre', 'Mujer')
      AND m.escenario_hecho NOT IN ('Sin Información', 'Otros')
    GROUP BY v.sexo, m.escenario_hecho
)
SELECT sexo, puesto, escenario_hecho, homicidios, pct_del_sexo
FROM escenarios
WHERE puesto <= 5
ORDER BY sexo, puesto;
'''
c6 = consultar(C6)
c6

,sexo,puesto,escenario_hecho,homicidios,pct_del_sexo
0,Hombre,1,Vía Pública,61805,59.7
1,Hombre,2,Vivienda,13116,12.7
2,Hombre,3,"Espacios Terrestres al Aire Libre (Bosque,Potr...",7749,7.5
3,Hombre,4,"Calle (Autopista,Avenida,Dentro de La Ciudad)",5278,5.1
4,Hombre,5,Lugares de Esparcimiento con Expendio de Alcohol,3052,2.9
5,Mujer,1,Vivienda,3401,37.4
6,Mujer,2,Vía Pública,3344,36.8
7,Mujer,3,"Espacios Terrestres al Aire Libre (Bosque,Potr...",738,8.1
8,Mujer,4,Lugares de Esparcimiento con Expendio de Alcohol,236,2.6
9,Mujer,5,"Calle (Autopista,Avenida,Dentro de La Ciudad)",229,2.5


**Interpretación.** Para los hombres el homicidio es un fenómeno del **espacio público**: la vía pública reúne el 59,7 % de los casos con escenario conocido y la vivienda solo el 12,7 %. Para las mujeres la **vivienda es el primer escenario (37,4 %)**, por encima de la vía pública (36,8 %) y casi tres veces la proporción observada en hombres. Junto con C5, esto indica que la prevención del homicidio de mujeres requiere acciones sobre violencia intrafamiliar y de pareja, no solo seguridad en la calle.

## Resumen
- **Tiempo:** nivel más alto desde 2021 (+16,9 %) y concentración en noches de fin de semana.
- **Territorio:** Valle, Antioquia y Bogotá concentran la mayoría de casos, pero el crecimiento está en Cauca, Magdalena, Bolívar y Atlántico.
- **Modalidad:** arma de fuego contra hombres jóvenes en la calle; en mujeres pesan más la vivienda y los mecanismos de contacto cercano.